# Assignment 2

In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Load the dataset
path = "/content/House Price India.csv"
df = pd.read_csv(path)

In [2]:
if 'Date' in df.columns:
    df.drop(columns=['Date'], inplace=True)

# Fill missing values
for col in df.columns:
    if df[col].dtype == 'O':  # Categorical
        df[col] = df[col].fillna(df[col].mode()[0])
    else:  # Numerical
        df[col] = df[col].fillna(df[col].median())


In [3]:
# Encode 'Label' column if it exists
if 'Label' in df.columns:
    le = LabelEncoder()
    df['Label'] = le.fit_transform(df['Label'])

# Splitting dataset into features (X) and target (y)
# We'll predict 'Price'
X = df.drop(columns=['Price'])
y = df['Price']

# Apply a log transformation to the target for better numerical stability
y = np.log1p(y)  # log(1+y) transformation

In [4]:
# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Normalize features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Build the simple ANN model using an explicit Input layer
model = Sequential([
    Input(shape=(X_train.shape[1],)),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(1)  # Output layer for regression
])

model.compile(optimizer='adam', loss='mse', metrics=['mae'])

# Train the model
history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=50, batch_size=32)

# Evaluate the model on the test set
test_loss, test_mae = model.evaluate(X_test, y_test)
print(f"Test MAE (in log scale): {test_mae}")

Epoch 1/50
366/366 ━━━━━━━━━━━━━━━━━━━━ 9s 8ms/step - loss: 68.8387 - mae: 6.7522 - val_loss: 4.3504 - val_mae: 1.2065
Epoch 2/50
366/366 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 2.0112 - mae: 1.0894 - val_loss: 2.6432 - val_mae: 0.9410
Epoch 3/50
366/366 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 1.2567 - mae: 0.8578 - val_loss: 1.6948 - val_mae: 0.7539
Epoch 4/50
366/366 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.7909 - mae: 0.6888 - val_loss: 1.0810 - val_mae: 0.6039
Epoch 5/50
366/366 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 0.4898 - mae: 0.5351 - val_loss: 0.6887 - val_mae: 0.4536
Epoch 6/50
366/366 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.2653 - mae: 0.3968 - val_loss: 0.4340 - val_mae: 0.3518
Epoch 7/50
366/366 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - loss: 0.1519 - mae: 0.2995 - val_loss: 0.2654 - val_mae: 0.2580
Epoch 8/50
366/366 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - loss: 0.0879 - mae: 0.2249 - val_loss: 0.1664 - val_mae: 0.2007
Epoch 9/50
366/366 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - l

In [5]:
# To interpret predictions in original scale, apply the inverse transformation
y_pred = np.expm1(model.predict(X_test))
print("Sample Predictions in original scale:", y_pred[:5])

92/92 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step
Sample Predictions in original scale: [[267665.12]
 [597236.9 ]
 [666351.75]
 [565876.5 ]
 [695088.25]]
